In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect('../security_logs.db')
cursor = conn.cursor()

In [ ]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS unified_events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp DATETIME,
    process TEXT,
    pid INTEGER,
    thread TEXT,
    type TEXT,
    subsystem TEXT,
    activity TEXT,
    raw_message TEXT,
    message TEXT
)
''')

conn.commit()

In [ ]:
import pandas as pd

df = pd.read_csv('../datasets/processed/unified_logs_processed.csv')

# Rebuild the derived table so rerunning the notebook does not duplicate rows.
cursor.execute('DELETE FROM unified_events')

for _, row in df.iterrows():
    cursor.execute('''
    INSERT INTO unified_events (timestamp, process, pid, thread, type, subsystem, activity, raw_message, message)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (row['timestamp'], row['process'], row['pid'], row['thread'], row['type'], row['subsystem'], row['activity'], row['raw_message'], row['message']))

conn.commit()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

In [ ]:
# determine hour of day, day of week, and whether it's a weekend
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'] >= 5

In [ ]:
# macOS aware off-hours detection: 8pm to 6am on weekdays, all day weekends
df['off_hours'] = df.apply(
    lambda x: 1 if (x['hour'] < 6 or x['hour'] > 20 or x['is_weekend']) else 0,
    axis=1
)

In [ ]:
# Determine process frequency
process_counts = df['process'].value_counts()

df['process_frequency'] = df['process'].map(process_counts)

In [ ]:
# Determine subsystem frequency
subsystem_counts = df['subsystem'].value_counts()

df['subsystem_frequency'] = df['subsystem'].map(subsystem_counts)

In [ ]:
# Determine if the log entry is a error
df['is_error'] = (
    df['type']
      .str.lower()
      .eq('error')
      .astype(int)
)

In [ ]:
# Determine if the log entry is a fault
df['is_fault'] = (
    df['type']
      .str.lower()
      .eq('fault')
      .astype(int)
)

In [ ]:
# macOS system processes commonly safe to ignore in anomaly detection
system_processes = [
    # core kernel / system
    'kernel_task',
    'kernel',
    'launchd',

    # power management
    'powerd',

    # graphics / UI system
    'WindowServer',

    # logging & diagnostics
    'logd',
    'syslogd',

    # security
    'trustd',
    'securityd',

    # user/session services
    'UserEventAgent',
    'cfprefsd',

    # app lifecycle / resource management
    'runningboardd',
    'containermanagerd',

    # notifications / system comms
    'notifyd',
    'distnoted'
]

df['is_system_process'] = df['process'].isin(system_processes).astype(int)

In [ ]:
# filter user / non-system processes
user_df = df[df['is_system_process'] == 0]

process_counts = user_df['process'].value_counts()

if len(process_counts) > 0:
    rare_threshold = process_counts.quantile(0.05)
    rare_processes = process_counts[process_counts <= rare_threshold].index
else:
    rare_processes = []

df['rare_process'] = df['process'].isin(rare_processes).astype(int)

In [ ]:
# first seen process time detection
first_seen = df.groupby('process')['timestamp'].min().reset_index()
first_seen.columns = ['process', 'first_seen_time']

df = df.merge(first_seen, on='process', how='left')

df['first_seen'] = (df['timestamp'] == df['first_seen_time']).astype(int)

In [ ]:
# session awareness detection (sleep/wake)
df = df.sort_values('timestamp')

df['session_id'] = (df['timestamp'].diff() > pd.Timedelta('1h')).cumsum()
df['first_seen_session'] = (
    df.groupby(['process', 'session_id']).cumcount() == 0
).astype(int)

In [ ]:
# burst detection: count of events in the last hour
df = df.sort_values(['process', 'timestamp']).copy()

# Create temporary count column
df['_temp'] = 1

# Rolling sum with proper grouping
result = (
    df.set_index('timestamp')
    .groupby('process', group_keys=False)['_temp']
    .rolling('1h')
    .sum()
)

df['count_last_hour'] = result.values
df = df.drop('_temp', axis=1)
df['count_last_hour'] = df['count_last_hour'].fillna(1)

In [ ]:
security_keywords = [
    "Sandbox",
    "TCC",
    "AMFI",
    "EndpointSecurity",
    "XProtect",
    "Security",
    "Firewall"
]

df["security_related"] = (
    df["subsystem"]
      .fillna("")
      .apply(
          lambda x: int(
              any(k.lower() in x.lower() for k in security_keywords)
          )
      )
)

In [ ]:
df.to_csv('../datasets/enriched/unified_logs_enriched.csv', index=False)